# AuditFlow — Load & Performance Testing

A configurable load-testing notebook for the AuditFlow audit-logging pipeline.
Supports state-of-the-art L&P testing patterns:

- **Multi-threaded** concurrent publishing
- **Linear ramp-up** (gradually increase load)
- **Duration-based** or **count-based** termination
- **Real-time metrics** (throughput, latency percentiles, error rate)
- **Configurable parameters** (threads, ramp-up, timeout, payload size)

## Prerequisites

Start the local stack from the repo root:

```bash
just up          # builds the JAR, all images, starts the stack
just status      # wait until all containers show "healthy"
```

## 1. Test Configuration

Adjust these parameters to control the load profile.

In [1]:
import requests
import threading
import time
import uuid
import json
import math
import sys
from dataclasses import dataclass, field
from datetime import datetime, timezone
from collections import defaultdict

# ── Endpoint ──────────────────────────────────────────────────
BACKEND = "http://localhost:8080/api/v1/audit/publish"

# ── Load Profile ──────────────────────────────────────────────
NUM_THREADS      = 5         # concurrent worker threads
RAMP_UP_SECONDS  = 2         # linear ramp-up: threads join one-by-one over this period
DURATION_SECONDS = 60        # stop after this many seconds (set 0 to ignore)
TOTAL_EVENTS     = 0         # stop after this many events (set 0 to ignore, use duration)

# When both DURATION_SECONDS > 0 and TOTAL_EVENTS > 0, whichever limit is hit first wins.
# If both are 0, the test runs for 10 seconds as a safety fallback.
if DURATION_SECONDS == 0 and TOTAL_EVENTS == 0:
    DURATION_SECONDS = 10
    print(f"Both duration and total are 0 — defaulting to {DURATION_SECONDS}s")

# ── Request Settings ──────────────────────────────────────────
HTTP_TIMEOUT     = 10        # per-request timeout in seconds
PAYLOAD_SIZE     = "small"   # "small" (~200B) | "medium" (~1KB) | "large" (~10KB)
CONTENT_TYPE     = "application/json"

# ── Event Template ───────────────────────────────────────────
TENANT_ID      = "LOAD-TEST"
SOURCE_SYSTEM  = "load-test-runner"

print(f"Configuration:")
print(f"  Threads:        {NUM_THREADS}")
print(f"  Ramp-up:        {RAMP_UP_SECONDS}s")
print(f"  Duration:       {DURATION_SECONDS}s" if DURATION_SECONDS > 0 else "  Duration:       ∞")
print(f"  Total events:   {TOTAL_EVENTS}" if TOTAL_EVENTS > 0 else "  Total events:   ∞")
print(f"  Payload size:   {PAYLOAD_SIZE}")
print(f"  Timeout:        {HTTP_TIMEOUT}s")

Configuration:
  Threads:        100
  Ramp-up:        2s
  Duration:       300s
  Total events:   ∞
  Payload size:   small
  Timeout:        10s


## 2. Payload Generator & Helpers

In [2]:
def generate_payload(event_id: str, seq: int, size: str = PAYLOAD_SIZE) -> dict:
    """Generate an audit event payload of the requested approximate size."""
    base = {
        "eventId":      event_id,
        "eventType":    "load.test.event",
        "sourceSystem": SOURCE_SYSTEM,
        "tenantId":     TENANT_ID,
        "extra": {
            "seq":       seq,
            "thread":    threading.current_thread().name,
            "timestamp": time.time(),
            "action_status": "SUCCESS",
        }
    }
    if size == "medium":
        base["extra"]["padding"] = "x" * 800
    elif size == "large":
        base["extra"]["padding"] = "x" * 9800
    return base


def percentile(sorted_values: list[float], p: float) -> float:
    """Compute the p-th percentile from a sorted list."""
    if not sorted_values:
        return 0.0
    k = (len(sorted_values) - 1) * (p / 100)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return sorted_values[int(k)]
    return sorted_values[f] * (c - k) + sorted_values[c] * (k - f)


def human_duration(seconds: float) -> str:
    """Format seconds into human-readable string."""
    if seconds < 1:
        return f"{seconds*1000:.0f}ms"
    return f"{seconds:.2f}s"


print("Helpers ready.")

Helpers ready.


## 3. Metrics Collector

In [3]:
@dataclass
class LoadTestMetrics:
    """Thread-safe metrics collector for the load test."""
    _lock: threading.Lock = field(default_factory=threading.Lock)
    _latencies: list = field(default_factory=list)
    _errors: int = 0
    _success: int = 0
    _start_time: float = 0.0
    _end_time: float = 0.0
    _status_codes: dict = field(default_factory=lambda: defaultdict(int))

    def start(self):
        self._start_time = time.monotonic()

    def stop(self):
        self._end_time = time.monotonic()

    def record(self, latency_ms: float, status_code: int, success: bool):
        with self._lock:
            self._latencies.append(latency_ms)
            self._status_codes[status_code] += 1
            if success:
                self._success += 1
            else:
                self._errors += 1

    def elapsed(self) -> float:
        end = self._end_time if self._end_time else time.monotonic()
        return end - self._start_time

    def total(self) -> int:
        return self._success + self._errors

    def summary(self) -> dict:
        elapsed = self.elapsed()
        sorted_lat = sorted(self._latencies)
        return {
            "total_requests":   self.total(),
            "success":          self._success,
            "errors":           self._errors,
            "error_rate_pct":   (self._errors / max(self.total(), 1)) * 100,
            "elapsed_seconds":  elapsed,
            "throughput_rps":   self.total() / max(elapsed, 0.001),
            "latency_min_ms":   percentile(sorted_lat, 0),
            "latency_p50_ms":   percentile(sorted_lat, 50),
            "latency_p90_ms":   percentile(sorted_lat, 90),
            "latency_p95_ms":   percentile(sorted_lat, 95),
            "latency_p99_ms":   percentile(sorted_lat, 99),
            "latency_max_ms":   percentile(sorted_lat, 100),
            "latency_mean_ms":  sum(sorted_lat) / max(len(sorted_lat), 1),
            "status_codes":     dict(self._status_codes),
        }

    def print_summary(self):
        s = self.summary()
        print("\n" + "=" * 60)
        print("  LOAD TEST RESULTS")
        print("=" * 60)
        print(f"  Total requests:    {s['total_requests']}")
        print(f"  Successful:        {s['success']}")
        print(f"  Errors:            {s['errors']}")
        print(f"  Error rate:        {s['error_rate_pct']:.2f}%")
        print(f"  Elapsed time:      {s['elapsed_seconds']:.2f}s")
        print(f"  Throughput:        {s['throughput_rps']:.1f} req/s")
        print("-" * 60)
        print(f"  Latency min:       {s['latency_min_ms']:.1f}ms")
        print(f"  Latency mean:      {s['latency_mean_ms']:.1f}ms")
        print(f"  Latency p50:       {s['latency_p50_ms']:.1f}ms")
        print(f"  Latency p90:       {s['latency_p90_ms']:.1f}ms")
        print(f"  Latency p95:       {s['latency_p95_ms']:.1f}ms")
        print(f"  Latency p99:       {s['latency_p99_ms']:.1f}ms")
        print(f"  Latency max:       {s['latency_max_ms']:.1f}ms")
        print("-" * 60)
        print(f"  Status codes:      {s['status_codes']}")
        print("=" * 60)

metrics = LoadTestMetrics()
print("Metrics collector ready.")

Metrics collector ready.


## 4. Worker Thread

In [4]:
# Shared stop signal — set to True to tell all workers to finish
_stop_event = threading.Event()
# Shared counter for total events across all threads (count-based mode)
_event_counter = 0
_counter_lock = threading.Lock()


def worker_fn(thread_name: str, start_delay: float, session: requests.Session):
    """A single worker thread: sends events until stop signal or counter limit."""
    global _event_counter
    # Ramp-up delay: each thread waits its turn before starting
    time.sleep(start_delay)
    seq = 0
    while not _stop_event.is_set():
        # Count-based termination
        if TOTAL_EVENTS > 0:
            with _counter_lock:
                if _event_counter >= TOTAL_EVENTS:
                    break
                _event_counter += 1
            seq += 1

        event_id = str(uuid.uuid4())
        payload = generate_payload(event_id, seq)

        t0 = time.monotonic()
        try:
            resp = session.post(BACKEND, json=payload, timeout=HTTP_TIMEOUT)
            latency_ms = (time.monotonic() - t0) * 1000
            metrics.record(latency_ms, resp.status_code, resp.ok)
        except requests.RequestException as e:
            latency_ms = (time.monotonic() - t0) * 1000
            metrics.record(latency_ms, 0, False)

    # If count-based, still count this thread's non-counter events
    if TOTAL_EVENTS == 0:
        pass  # no counter to update


print("Worker function ready.")

Worker function ready.


## 5. Run Load Test

In [5]:
def run_load_test():
    """Execute the load test with the configured parameters."""
    global _event_counter, _stop_event
    _event_counter = 0
    _stop_event.clear()

    # Pre-flight: verify backend is reachable
    try:
        r = requests.get("http://localhost:8080/actuator/health", timeout=5)
        if r.json().get("status") != "UP":
            print("ERROR: Backend health is not UP. Is the stack running?")
            return
    except Exception as e:
        print(f"ERROR: Cannot reach backend: {e}")
        print("Run: just up")
        return

    print(f"Starting load test: {NUM_THREADS} threads, ramp-up {RAMP_UP_SECONDS}s")
    print(f"Mode: {'duration' if DURATION_SECONDS > 0 else 'count'}-based")
    print("-" * 60)

    metrics.start()
    t_start = time.monotonic()

    # Create threads with staggered start for ramp-up
    threads = []
    session = requests.Session()
    for i in range(NUM_THREADS):
        delay = (RAMP_UP_SECONDS * i) / max(NUM_THREADS - 1, 1) if RAMP_UP_SECONDS > 0 else 0
        t = threading.Thread(
            target=worker_fn,
            args=(f"worker-{i}", delay, session),
            name=f"worker-{i}",
            daemon=True
        )
        threads.append(t)

    # Start all threads
    for t in threads:
        t.start()

    # Monitor loop: print periodic updates and handle duration-based stop
    last_print = time.monotonic()
    while not _stop_event.is_set():
        time.sleep(1)
        now = time.monotonic()
        elapsed = now - t_start

        # Duration-based termination
        if DURATION_SECONDS > 0 and elapsed >= DURATION_SECONDS:
            _stop_event.set()
            break

        # Count-based termination
        if TOTAL_EVENTS > 0 and _event_counter >= TOTAL_EVENTS:
            _stop_event.set()
            break

        # Periodic progress (every 5 seconds)
        if now - last_print >= 5:
            s = metrics.summary()
            remaining = ""
            if DURATION_SECONDS > 0:
                remaining = f", {max(DURATION_SECONDS - elapsed, 0):.0f}s left"
            elif TOTAL_EVENTS > 0:
                remaining = f", {max(TOTAL_EVENTS - _event_counter, 0)} events left"
            print(f"  [{elapsed:5.1f}s] {s['total_requests']:>6} reqs | {s['throughput_rps']:>8.1f} req/s | p50={s['latency_p50_ms']:.0f}ms p99={s['latency_p99_ms']:.0f}ms | errors={s['errors']}{remaining}")
            last_print = now

    # Wait for all threads to finish
    for t in threads:
        t.join(timeout=5)

    metrics.stop()
    session.close()

    metrics.print_summary()


run_load_test()

Starting load test: 100 threads, ramp-up 2s
Mode: duration-based
------------------------------------------------------------
  [  5.1s]   4493 reqs |    889.4 req/s | p50=79ms p99=243ms | errors=0, 295s left
  [ 10.1s]   9438 reqs |    933.7 req/s | p50=86ms p99=243ms | errors=0, 290s left
  [ 15.2s]  13771 reqs |    907.7 req/s | p50=90ms p99=309ms | errors=0, 285s left
  [ 20.2s]  17805 reqs |    880.7 req/s | p50=95ms p99=316ms | errors=0, 280s left
  [ 25.3s]  21740 reqs |    860.5 req/s | p50=96ms p99=349ms | errors=0, 275s left
  [ 30.4s]  26312 reqs |    865.5 req/s | p50=97ms p99=340ms | errors=0, 270s left
  [ 35.7s]  30537 reqs |    856.2 req/s | p50=98ms p99=339ms | errors=0, 264s left
  [ 40.7s]  34742 reqs |    852.9 req/s | p50=99ms p99=338ms | errors=0, 259s left
  [ 45.8s]  38792 reqs |    847.1 req/s | p50=100ms p99=341ms | errors=0, 254s left
  [ 50.9s]  42906 reqs |    843.5 req/s | p50=101ms p99=345ms | errors=0, 249s left
  [ 56.0s]  46475 reqs |    830.3 req/s | 

## 6. Quick Re-run

Change parameters in Section 1 and re-execute Section 5, or use this convenience cell for common profiles.

In [10]:
# Common test profiles — uncomment one and run

# ── Baseline: low load, short duration ────────────────────────
# NUM_THREADS, RAMP_UP_SECONDS, DURATION_SECONDS, TOTAL_EVENTS, PAYLOAD_SIZE = 2, 1, 10, 0, "small"

# ── Sustained: moderate load ──────────────────────────────────
# NUM_THREADS, RAMP_UP_SECONDS, DURATION_SECONDS, TOTAL_EVENTS, PAYLOAD_SIZE = 10, 3, 30, 0, "small"

# ── Burst: high threads, short burst ─────────────────────────
# NUM_THREADS, RAMP_UP_SECONDS, DURATION_SECONDS, TOTAL_EVENTS, PAYLOAD_SIZE = 50, 1, 10, 0, "small"

# ── Count-based: exactly N events ────────────────────────────
# NUM_THREADS, RAMP_UP_SECONDS, DURATION_SECONDS, TOTAL_EVENTS, PAYLOAD_SIZE = 10, 2, 0, 1000, "small"

# ── Large payload stress ─────────────────────────────────────
# NUM_THREADS, RAMP_UP_SECONDS, DURATION_SECONDS, TOTAL_EVENTS, PAYLOAD_SIZE = 5, 2, 15, 0, "large"

# run_load_test()

## 7. Post-Test Verification

Verify the pipeline processed the events correctly after the load test.

In [14]:
ACTUATOR = "http://localhost:8080/actuator"

print("Backend health:")
try:
    r = requests.get(f"{ACTUATOR}/health", timeout=5)
    print(f"  Status: {r.json().get('status', 'UNKNOWN')}")
except Exception as e:
    print(f"  Error: {e}")

print("\nConsumer metrics:")
try:
    r = requests.get(f"{ACTUATOR}/metrics/auditflow.consumer.events.processed", timeout=5)
    value = r.json()["measurements"][0]["value"]
    print(f"  events.processed = {value}")
except Exception as e:
    print(f"  Could not read metric: {e}")

print("\nRabbitMQ queue depth:")
try:
    queue = "labs64-audit-topic.labs64.io-auditflow"
    r = requests.get(
        f"http://localhost:15673/api/queues/%2F/{queue}",
        auth=("guest", "guest"),
        timeout=5
    )
    ready = r.json()["messages_ready"]
    print(f"  messages_ready = {ready} (should be 0)")
except Exception as e:
    print(f"  Could not check queue: {e}")

Backend health:
  Status: UP

Consumer metrics:
  events.processed = 293396.0

RabbitMQ queue depth:
  messages_ready = 146937 (should be 0)


## Next Steps

- **Increase load gradually**: start with 10 threads, work up to 100+ to find the saturation point.
- **Monitor backend logs**: `just log backend` — watch for errors or queue depth growth.
- **Observe metrics**: `just up obs` then check Grafana dashboards for real-time throughput and latency.
- **Tune JVM**: adjust `-Xmx` / `-Xms` in `JAVA_OPTS` if you see memory pressure under load.
- **Tune pipeline**: adjust `concurrency`, `prefetch`, and `maxConcurrency` in `application.yml`.